# Общий отчёт свипа: собрать прогоны всех участников

1. Справа **Accelerator → None** (GPU не нужна).
2. **Add Input → Notebooks**: добавить готовые ноутбуки всех участников (свои — во вкладке Your Work,
   чужие — по ссылке; их владелец должен дать доступ через Share или сделать Public).
3. **Run All**. В `/kaggle/working` появится `sweep_report.pdf`.

In [ ]:
REPO = "https://github.com/NordManul/Project_Transformers.git"

import glob, json, os, shutil, subprocess, time
T0 = time.time()
WORK = "/kaggle/working"
CODE = f"{WORK}/Project_Transformers"
DATA = f"{WORK}/data/owt"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
os.chdir(CODE)
print("код:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
import torch
print("torch", torch.__version__, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

ALL = f"{WORK}/all"
os.makedirs(f"{ALL}/runs", exist_ok=True)
for m in sorted(glob.glob("/kaggle/input/**/runs/*/runs/*/metrics.jsonl", recursive=True)):
    src = os.path.dirname(m)
    name = os.path.basename(src)
    if '"type": "final"' not in open(m, encoding="utf-8").read():
        print("не закончен, пропускаю:", src)
        continue
    dst = f"{ALL}/runs/{name}"
    if os.path.exists(dst):
        print("уже есть такой прогон, пропускаю дубль:", src)
        continue
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("*.pt"))   # веса не нужны
    cfg = json.load(open(f"{dst}/config.json", encoding="utf-8"))
    print(f"{name:32s} lr {cfg['args']['lr']:<8g} сид {cfg['args'].get('seed', 42):<4} код {cfg.get('git_commit', '?')}")

In [ ]:
!python ngpt_vs_gpt.py report --out {ALL} > /dev/null
!python make_pdf_report.py {ALL}/report --out /kaggle/working/sweep_report.pdf
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{ALL}/report/report.md", encoding="utf-8").read()))
display(Image(f"{ALL}/report/ngpt_vs_gpt.png"))